# ⚡ Manga Studio Cloud (Google Colab GPU)
### Execute o Manga Studio gratuitamente na nuvem da Google!
Todo o processamento pesado de imagens e IA roda nos servidores da Google com **0% de uso do seu PC**.

---
### 🚀 Como usar:
1. No menu superior do Colab, vá em **Ambiente de execução** > **Alterar tipo de ambiente de execução** e certifique-se de que está em **T4 GPU**.
2. Marque a opção de **salvar_no_google_drive** caso queira salvar seus mangás direto no seu Drive.
3. Clique no botão de Play `[ ▶ ]` na célula abaixo.
4. Em cerca de 30 segundos, o link seguro da Cloudflare (ex: `https://xxxx.trycloudflare.com`) será exibido.
5. Abra o link no navegador para usar o **Manga Studio completo** no computador ou no celular!

In [ ]:
# @title ▶ Iniciar Manga Studio na Nuvem (Clique no Play)
# @markdown **Deseja salvar e carregar mangás traduzidos direto do seu Google Drive?**
salvar_no_google_drive = True # @param {type:"boolean"}

import os, subprocess, time, re, sys

# 1. Garante que o diretório inicial seja sempre /content
try:
    os.chdir('/content')
except Exception:
    pass

# 2. Conectar Google Drive se marcado
if salvar_no_google_drive:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        os.makedirs('/content/drive/MyDrive/MANGAS [PT-BR]', exist_ok=True)
        os.makedirs('/content/drive/MyDrive/MANGAS', exist_ok=True)
        print("📁 Google Drive conectado com sucesso! Obras salvas em: Google Drive > MANGAS [PT-BR]")
    except Exception as e:
        print(f"[!] Aviso Google Drive: {e}")

# 3. Baixar ou atualizar o Manga Studio
print("Atualizando / Baixando o Manga Studio do GitHub...")
if os.path.exists('/content/manga-translator/.git'):
    os.chdir('/content/manga-translator')
    !git pull
else:
    os.chdir('/content')
    !rm -rf /content/manga-translator
    !git clone https://github.com/Jos9011/SKILL-TRADUCAO-MANGA.git /content/manga-translator
    os.chdir('/content/manga-translator')

print("Verificando dependências de OCR e tradução...")
!pip install -q flask pillow opencv-python-headless rapidocr_onnxruntime deep_translator pyphen wordninja

print("Configurando túnel seguro Cloudflare...")
!which cloudflared > /dev/null 2>&1 || (wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1)

# Encerra qualquer processo anterior para liberar a porta 5050
!pkill -f 'studio/server.py' || true
!pkill -f 'cloudflared tunnel' || true
time.sleep(1)

# Inicia o servidor Flask em segundo plano
p_srv = subprocess.Popen([sys.executable, 'studio/server.py'])
time.sleep(3)

# Inicia túnel Cloudflare para gerar o link público
p_tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:5050'], stderr=subprocess.PIPE, text=True)

print('\n' + '='*60)
print('   🚀 MANGA STUDIO INICIANDO NA NUVEM DA GOOGLE...')
print('='*60)

for line in p_tunnel.stderr:
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        url = match.group(0)
        print(f'\n✅ MANGA STUDIO PRONTO E OPERACIONAL!')
        print(f'👉 LINK DE ACESSO: {url}')
        print('\n(Você pode abrir este link no computador ou celular. Deixe esta aba do Colab aberta enquanto traduz.)\n')
        break


In [ ]:
# @title 📁 (Opcional) Conectar ao seu Google Drive Manualmente
# Execute esta célula se quiser conectar o Google Drive separadamente
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/MANGAS [PT-BR]', exist_ok=True)
os.makedirs('/content/drive/MyDrive/MANGAS', exist_ok=True)
print('Google Drive conectado com sucesso! Suas pastas estão em /content/drive/MyDrive/MANGAS [PT-BR]')
